# lumina-bg — Video Background Replacement + Relighting

> Replace your video background and relight yourself — no green screen, no studio.  
> Powered by **IC-Light + Stable Diffusion 1.5**, runs free on Google Colab.

---

**Pipeline:**  
1. Extract frames from video  
2. Remove background (AI)  
3. Generate new background from text prompt (SD 1.5)  
4. Relight person to match new environment (IC-Light)  
5. Export final video with original audio  

**Runtime:** Go to `Runtime > Change runtime type > GPU (T4)`

## 1. Montar Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# Criar estrutura de pastas
BASE_DIR = "/content/drive/MyDrive/iclight_pipeline"
dirs = [
    f"{BASE_DIR}/input",
    f"{BASE_DIR}/frames/raw",
    f"{BASE_DIR}/frames/nobg",
    f"{BASE_DIR}/background",
    f"{BASE_DIR}/relit",
    f"{BASE_DIR}/output",
]
for d in dirs:
    os.makedirs(d, exist_ok=True)

print("Google Drive montado!")
print(f"Pasta do projeto: {BASE_DIR}")

## 2. Instalar dependencias

In [ ]:
%%bash
apt-get update -qq && apt-get install -y -qq ffmpeg > /dev/null 2>&1
echo "ffmpeg instalado"

# PyTorch (ja vem no Colab, mas garantir CUDA)
pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118 2>/dev/null

# rembg com GPU
pip install -q rembg[gpu] onnxruntime-gpu 2>/dev/null

# Diffusers + transformers
pip install -q diffusers transformers accelerate safetensors 2>/dev/null

# Utilidades
pip install -q opencv-python Pillow numpy requests tqdm huggingface_hub 2>/dev/null

echo "Dependencias instaladas!"

## 3. Clonar ComfyUI + IC-Light

In [ ]:
import os
import subprocess

# ComfyUI
if not os.path.exists("/content/ComfyUI"):
    print("Clonando ComfyUI...")
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/comfyanonymous/ComfyUI",
                    "/content/ComfyUI"], check=True)
    subprocess.run(["pip", "install", "-q", "-r",
                    "/content/ComfyUI/requirements.txt"],
                   check=True, capture_output=True)
    print("ComfyUI instalado!")
else:
    print("ComfyUI ja existe, pulando.")

# IC-Light
if not os.path.exists("/content/IC-Light"):
    print("Clonando IC-Light...")
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/lllyasviel/IC-Light",
                    "/content/IC-Light"], check=True)
    req_file = "/content/IC-Light/requirements.txt"
    if os.path.exists(req_file):
        subprocess.run(["pip", "install", "-q", "-r", req_file],
                       check=True, capture_output=True)
    print("IC-Light instalado!")
else:
    print("IC-Light ja existe, pulando.")

## 4. Download dos modelos (SD 1.5 + IC-Light)

In [ ]:
from huggingface_hub import hf_hub_download
import os

# SD 1.5 checkpoint para ComfyUI
ckpt_dir = "/content/ComfyUI/models/checkpoints"
ckpt_file = os.path.join(ckpt_dir, "v1-5-pruned-emaonly.safetensors")
if not os.path.exists(ckpt_file):
    print("Baixando SD 1.5...")
    hf_hub_download(
        repo_id="runwayml/stable-diffusion-v1-5",
        filename="v1-5-pruned-emaonly.safetensors",
        local_dir=ckpt_dir,
    )
    print("SD 1.5 baixado!")
else:
    print("SD 1.5 ja existe, pulando.")

# IC-Light weights (foreground conditioned)
icl_dir = "/content/IC-Light/models"
os.makedirs(icl_dir, exist_ok=True)
icl_file = os.path.join(icl_dir, "iclight_sd15_fc.safetensors")
if not os.path.exists(icl_file):
    print("Baixando IC-Light weights...")
    hf_hub_download(
        repo_id="lllyasviel/ic-light",
        filename="iclight_sd15_fc.safetensors",
        local_dir=icl_dir,
    )
    print("IC-Light weights baixado!")
else:
    print("IC-Light weights ja existe, pulando.")

print("\nModelos prontos!")

## 5. Upload do video + fundo (opcional)

**Video:** obrigatorio — eh o seu video original.  
**Fundo:** opcional — se voce ja tem uma imagem de fundo pronta (foto, print, etc), faz upload aqui tambem. Se nao tiver, a IA gera um pra voce na etapa 6.

In [ ]:
import os
import shutil
from google.colab import files
from PIL import Image

BASE_DIR = "/content/drive/MyDrive/iclight_pipeline"
INPUT_DIR = f"{BASE_DIR}/input"
VIDEO_PATH = f"{INPUT_DIR}/video.mp4"
BG_CUSTOM_PATH = f"{BASE_DIR}/background/bg_custom.png"

# ═══════════════════════════════════════════════════════════════
# UPLOAD DO VIDEO
# ═══════════════════════════════════════════════════════════════
print("=" * 50)
print("  UPLOAD DO VIDEO")
print("=" * 50)

if os.path.exists(VIDEO_PATH):
    size_mb = os.path.getsize(VIDEO_PATH) / (1024 * 1024)
    print(f"Video encontrado no Drive: {VIDEO_PATH} ({size_mb:.1f} MB)")
    print("Pulando upload.")
else:
    print("Nenhum video encontrado no Drive. Faca upload abaixo:")
    uploaded = files.upload()
    if uploaded:
        filename = list(uploaded.keys())[0]
        shutil.move(filename, VIDEO_PATH)
        print(f"Video salvo em: {VIDEO_PATH}")
    else:
        print("ERRO: Nenhum arquivo enviado!")

# Mostrar info do video
if os.path.exists(VIDEO_PATH):
    import subprocess, json
    probe = subprocess.run(
        ["ffprobe", "-v", "error", "-select_streams", "v:0",
         "-show_entries", "stream=width,height,r_frame_rate,nb_frames",
         "-show_entries", "format=duration",
         "-of", "json", VIDEO_PATH],
        capture_output=True, text=True
    )
    info = json.loads(probe.stdout)
    stream = info["streams"][0]
    duration = float(info["format"]["duration"])
    fps_raw = stream["r_frame_rate"]
    num, den = map(int, fps_raw.split("/"))
    fps = round(num / den, 1)
    frames_est = int(duration * fps)
    print(f"\nInfo: {stream['width']}x{stream['height']} @ {fps}fps")
    print(f"Duracao: {duration:.1f}s (~{frames_est} frames)")
    print(f"Tempo estimado: ~{frames_est * 1.8 / 60:.0f} min")

# ═══════════════════════════════════════════════════════════════
# UPLOAD DO FUNDO (OPCIONAL)
# ═══════════════════════════════════════════════════════════════
print("\n" + "=" * 50)
print("  FUNDO PROPRIO (OPCIONAL)")
print("=" * 50)
print("Se voce tem uma imagem de fundo pronta, faca upload agora.")
print("Se nao tiver, so clique em 'Cancel' ou nao envie nada —")
print("a IA vai gerar um fundo pra voce na proxima celula.\n")

USAR_FUNDO_PROPRIO = False

try:
    uploaded_bg = files.upload()
    if uploaded_bg:
        bg_filename = list(uploaded_bg.keys())[0]
        # Salvar e converter pra PNG
        img = Image.open(bg_filename).convert("RGB")
        img.save(BG_CUSTOM_PATH)
        os.remove(bg_filename)
        USAR_FUNDO_PROPRIO = True
        print(f"\nFundo proprio salvo: {BG_CUSTOM_PATH}")
        print(f"Tamanho: {img.size[0]}x{img.size[1]}")
        print(">> A etapa de geracao por IA sera PULADA (mais rapido!)")
    else:
        print("Nenhum fundo enviado — a IA vai gerar um.")
except Exception:
    print("Nenhum fundo enviado — a IA vai gerar um.")

## 6. Configurar prompt e parametros

O **prompt** serve pra duas coisas:
- Se voce **NAO** enviou fundo proprio: a IA gera o fundo a partir desse texto
- Se voce **enviou** fundo proprio: o prompt ainda eh usado pro IC-Light ajustar a iluminacao

Exemplos de prompts:
- `"modern studio with soft blue ambient lighting, cinematic"`
- `"tropical beach at sunset, warm golden light"`
- `"cozy living room, fireplace, warm lighting"`
- `"futuristic neon city at night, cyberpunk"`

In [ ]:
#@title Configuracao da Pipeline { display-mode: "form" }

#@markdown ### Prompt do fundo
PROMPT = "modern studio with soft blue ambient lighting, cinematic" #@param {type:"string"}

#@markdown ### Parametros avancados
STEPS = 25 #@param {type:"slider", min:10, max:50, step:5}
SEED = 42 #@param {type:"integer"}
CRF = 18 #@param {type:"slider", min:10, max:35, step:1}
CFG_BG = 7.0 #@param {type:"slider", min:1.0, max:15.0, step:0.5}
CFG_RELIGHT = 2.0 #@param {type:"slider", min:1.0, max:5.0, step:0.5}

print(f"Prompt: {PROMPT}")
print(f"Steps: {STEPS} | Seed: {SEED} | CRF: {CRF}")
print(f"CFG background: {CFG_BG} | CFG relight: {CFG_RELIGHT}")

## 7. Executar pipeline

In [ ]:
import sys
import os
import time
import json
import shutil
import torch
from PIL import Image

# Verificar GPU
if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU nao encontrada! Va em Runtime > Change runtime type > GPU (T4)"
    )
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_mem / (1024**3)
print(f"GPU: {gpu_name} ({vram_gb:.1f} GB VRAM)")

# Clonar o repo com os agentes
REPO_DIR = "/content/lumina-bg"
if os.path.exists(f"{REPO_DIR}/agentes"):
    sys.path.insert(0, REPO_DIR)
elif os.path.exists("/content/agentes"):
    sys.path.insert(0, "/content")
else:
    print("Clonando repositorio lumina-bg...")
    import subprocess
    subprocess.run([
        "git", "clone", "--depth", "1",
        "https://github.com/rob-d3v/background-magic-free",
        REPO_DIR
    ], check=False)
    if os.path.exists(f"{REPO_DIR}/agentes"):
        sys.path.insert(0, REPO_DIR)
    else:
        raise RuntimeError(
            "Nao foi possivel encontrar os modulos agentes/. "
            "Clone o repo ou copie a pasta agentes/ para /content/"
        )

# Paths
BASE_DIR = "/content/drive/MyDrive/iclight_pipeline"
VIDEO_PATH = f"{BASE_DIR}/input/video.mp4"
FRAMES_RAW = f"{BASE_DIR}/frames/raw"
FRAMES_NOBG = f"{BASE_DIR}/frames/nobg"
FRAMES_RELIT = f"{BASE_DIR}/relit"
BG_OUTPUT = f"{BASE_DIR}/background/bg.png"
BG_CUSTOM_PATH = f"{BASE_DIR}/background/bg_custom.png"
OUTPUT_VIDEO = f"{BASE_DIR}/output/video_final.mp4"
LOG_PATH = f"{BASE_DIR}/pipeline_log.json"

if not os.path.exists(VIDEO_PATH):
    raise FileNotFoundError(f"Video nao encontrado em {VIDEO_PATH}")

# Detectar modo de fundo
usar_fundo_proprio = 'USAR_FUNDO_PROPRIO' in dir() and USAR_FUNDO_PROPRIO
if not usar_fundo_proprio:
    usar_fundo_proprio = os.path.exists(BG_CUSTOM_PATH)

pipeline_log = {"etapas": [], "erros_total": 0}
pipeline_start = time.time()

print("=" * 60)
print("  IC-Light Video Pipeline — lumina-bg")
if usar_fundo_proprio:
    print("  Modo: FUNDO PROPRIO (pula geracao por IA)")
else:
    print("  Modo: FUNDO GERADO POR IA")
print("=" * 60)

# ─── ETAPA 1 — Extracao de Frames ────────────────────────────
print("\n[1/5] Extraindo frames...")
from agentes.extracao import extrair_frames
meta = extrair_frames(VIDEO_PATH, FRAMES_RAW)
print(f"    {meta['total_frames']} frames @ {meta['fps']}fps — {meta['width']}x{meta['height']}")
pipeline_log["etapas"].append({"etapa": "extracao", **meta})

# ─── ETAPA 2 — Remocao de Fundo ──────────────────────────────
print("\n[2/5] Removendo fundo (rembg GPU)...")
from agentes.remocao import remover_fundo
result_rembg = remover_fundo(FRAMES_RAW, FRAMES_NOBG, log_path=LOG_PATH)
pipeline_log["etapas"].append({"etapa": "remocao_fundo", **result_rembg})
pipeline_log["erros_total"] += result_rembg["erros"]

# ─── ETAPA 3 — Fundo ─────────────────────────────────────────
if usar_fundo_proprio:
    print(f"\n[3/5] Usando fundo proprio...")
    bg_img = Image.open(BG_CUSTOM_PATH).convert("RGB")
    bg_img = bg_img.resize((meta["width"], meta["height"]), Image.LANCZOS)
    bg_img.save(BG_OUTPUT)
    print(f"    Fundo redimensionado para {meta['width']}x{meta['height']}")
    pipeline_log["etapas"].append({"etapa": "fundo", "modo": "proprio"})
else:
    print("\n[3/5] Gerando fundo com Stable Diffusion 1.5...")
    from agentes.geracao_fundo import iniciar_comfyui, gerar_fundo
    comfy_proc = iniciar_comfyui()
    try:
        gerar_fundo(
            prompt=PROMPT,
            width=meta["width"],
            height=meta["height"],
            output_path=BG_OUTPUT,
            steps=STEPS,
            cfg=CFG_BG,
            seed=SEED,
        )
        pipeline_log["etapas"].append({"etapa": "fundo", "modo": "ia", "status": "ok"})
    finally:
        comfy_proc.terminate()

# ─── ETAPA 4 — Relighting com IC-Light ────────────────────────
print("\n[4/5] Aplicando relighting com IC-Light...")
from agentes.relighting import carregar_iclight, aplicar_relighting
pipe = carregar_iclight()
result_relight = aplicar_relighting(
    pipe=pipe,
    frames_nobg_dir=FRAMES_NOBG,
    background_path=BG_OUTPUT,
    output_dir=FRAMES_RELIT,
    prompt=PROMPT,
    steps=STEPS,
    cfg=CFG_RELIGHT,
    seed=SEED,
    log_path=LOG_PATH,
)
pipeline_log["etapas"].append({"etapa": "relighting", **result_relight})
pipeline_log["erros_total"] += result_relight["erros"]

# Liberar VRAM
del pipe
torch.cuda.empty_cache()

# ─── ETAPA 5 — Exportacao ─────────────────────────────────────
print("\n[5/5] Exportando video final...")
from agentes.exportacao import exportar_video
result_export = exportar_video(
    frames_dir=FRAMES_RELIT,
    video_original=VIDEO_PATH,
    output_path=OUTPUT_VIDEO,
    fps=meta["fps"],
    crf=CRF,
)
pipeline_log["etapas"].append({"etapa": "exportacao", **result_export})

# ─── Log final ────────────────────────────────────────────────
pipeline_log["tempo_total_s"] = round(time.time() - pipeline_start, 2)
with open(LOG_PATH, "w") as f:
    json.dump(pipeline_log, f, indent=2)

print("\n" + "=" * 60)
print(f"  Concluido! Video salvo em: {OUTPUT_VIDEO}")
print(f"  Tempo total: {pipeline_log['tempo_total_s']:.0f}s ({pipeline_log['tempo_total_s']/60:.1f} min)")
if pipeline_log["erros_total"] > 0:
    print(f"  AVISO: {pipeline_log['erros_total']} frame(s) com erro — ver log")
print("=" * 60)

## 8. Preview (antes / depois)

In [ ]:
import os
import matplotlib.pyplot as plt
from PIL import Image

BASE_DIR = "/content/drive/MyDrive/iclight_pipeline"
FRAMES_RAW = f"{BASE_DIR}/frames/raw"
FRAMES_NOBG = f"{BASE_DIR}/frames/nobg"
FRAMES_RELIT = f"{BASE_DIR}/relit"
BG_OUTPUT = f"{BASE_DIR}/background/bg.png"

# Pegar frame do meio
frames_raw = sorted([f for f in os.listdir(FRAMES_RAW) if f.endswith(".png")])
mid_idx = len(frames_raw) // 2
mid_frame = frames_raw[mid_idx]

fig, axes = plt.subplots(1, 4, figsize=(20, 5))

# Original
img_raw = Image.open(os.path.join(FRAMES_RAW, mid_frame))
axes[0].imshow(img_raw)
axes[0].set_title("Original")
axes[0].axis("off")

# Sem fundo
nobg_path = os.path.join(FRAMES_NOBG, mid_frame)
if os.path.exists(nobg_path):
    img_nobg = Image.open(nobg_path)
    axes[1].imshow(img_nobg)
    axes[1].set_title("Sem fundo")
else:
    axes[1].text(0.5, 0.5, "N/A", ha="center", va="center")
    axes[1].set_title("Sem fundo")
axes[1].axis("off")

# Fundo gerado
if os.path.exists(BG_OUTPUT):
    img_bg = Image.open(BG_OUTPUT)
    axes[2].imshow(img_bg)
    axes[2].set_title("Fundo gerado (SD)")
else:
    axes[2].text(0.5, 0.5, "N/A", ha="center", va="center")
    axes[2].set_title("Fundo gerado")
axes[2].axis("off")

# Resultado final (relit)
relit_path = os.path.join(FRAMES_RELIT, mid_frame)
if os.path.exists(relit_path):
    img_relit = Image.open(relit_path)
    axes[3].imshow(img_relit)
    axes[3].set_title("Resultado (IC-Light)")
else:
    axes[3].text(0.5, 0.5, "N/A", ha="center", va="center")
    axes[3].set_title("Resultado")
axes[3].axis("off")

plt.suptitle(f"Frame {mid_idx + 1}/{len(frames_raw)}", fontsize=14)
plt.tight_layout()
plt.show()

## 9. Download do video final

In [ ]:
import os
from google.colab import files
from IPython.display import HTML

OUTPUT_VIDEO = "/content/drive/MyDrive/iclight_pipeline/output/video_final.mp4"

if os.path.exists(OUTPUT_VIDEO):
    size_mb = os.path.getsize(OUTPUT_VIDEO) / (1024 * 1024)
    print(f"Video final: {size_mb:.1f} MB")
    print(f"Salvo em: {OUTPUT_VIDEO}")
    print("\nO video tambem esta no seu Google Drive em:")
    print("  Google Drive > iclight_pipeline > output > video_final.mp4")
    print("\nClique abaixo para baixar:")
    files.download(OUTPUT_VIDEO)
else:
    print("ERRO: Video final nao encontrado. Execute a celula 7 primeiro.")